In [ ]:
import rasterio                  
import rasterio.mask
import rasterio.warp             
import rasterio.merge
from rasterio.transform import rowcol
import fiona                     
import pyproj                    
import geopandas as gps
import pandas as pd
import shapely
from shapely.geometry import box, Point
import json

import numpy as np              
import time
import os
from PIL import Image
import PIL.ImageDraw
from core.visualize import display_images
from core.frame_info import image_normalize

import matplotlib.pyplot as plt  
%matplotlib inline
from tqdm import tqdm_notebook as tqdm
import warnings                 
warnings.filterwarnings("ignore")

%reload_ext autoreload
%autoreload 2 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [ ]:
# Required configurations (including the input and output paths) are stored in a separate file (such as config/Preprocessing.py)
# Please provide required info in the file before continuing with this notebook. 

from config import Preprocessing
# In case you are using a different folder name such as configLargeCluster, then you should import from the respective folder 
# Eg. from configLargeCluster import Preprocessing

config = Preprocessing.Configuration()


In [ ]:
#Read the training area and training polygons
trainingArea = gps.read_file(os.path.join(config.training_base_dir, config.training_area_fn))
trainingPolygon = gps.read_file(os.path.join(config.training_base_dir, config.training_polygon_fn))

print(f'Read a total of {trainingPolygon.shape[0]} object polygons and {trainingArea.shape[0]} training areas.')
print(f'Polygons will be assigned to training areas in the next steps.')

In [ ]:
# Check if the training areas and the training polygons have the same CRS
if trainingArea.crs != trainingPolygon.crs:
    print('Training area CRS does not match training_polygon CRS. Converting...')
    targetCRS = trainingPolygon.crs
    trainingArea = trainingArea.to_crs(targetCRS)

print(trainingPolygon.crs)
print(trainingArea.crs)
assert trainingPolygon.crs == trainingArea.crs


In [ ]:
# Assign serial IDs to training areas
trainingArea['id'] = range(trainingArea.shape[0])
trainingArea

In [ ]:
def calculateBoundaryWeight(polygonsInArea, scale_polygon=1.5, output_plot=True): 
    '''
    For each polygon, create a weighted boundary where the weights of shared/close boundaries are higher than weights of solitary boundaries.
    '''
    # If there are no polygons in the area, return an empty GeoDataFrame
    if not polygonsInArea:
        print("No polygons in area. Returning an empty GeoDataFrame.")
        return gps.GeoDataFrame({})

    tempPolygonDf = pd.DataFrame({'geometry': polygonsInArea})
    tempPolygonDf = gps.GeoDataFrame(tempPolygonDf)
    tempPolygonDf.set_geometry("geometry", inplace=True)  # Set the geometry column

    new_c = []
    # For each polygon in area, scale and compare with other polygons
    for i in tqdm(range(len(tempPolygonDf))):
        pol1 = gps.GeoSeries(tempPolygonDf.iloc[i][0])
        sc = pol1.scale(xfact=scale_polygon, yfact=scale_polygon, zfact=scale_polygon, origin='center')
        scc = pd.DataFrame({'geometry': [sc[0]]})
        scc = gps.GeoDataFrame(pd.concat([scc]*len(tempPolygonDf), ignore_index=True))
        scc.set_geometry("geometry", inplace=True)  # Set the geometry column

        pol2 = gps.GeoDataFrame(tempPolygonDf[~tempPolygonDf.index.isin([i])])
        pol2 = gps.GeoDataFrame(pol2.scale(xfact=scale_polygon, yfact=scale_polygon, zfact=scale_polygon, origin='center'))
        pol2.columns = ['geometry']
        pol2.set_geometry("geometry", inplace=True)  # Set the geometry column

        ints = scc.intersection(pol2)
        for k in range(len(ints)):
            if ints.iloc[k] is not None and not ints.iloc[k].is_empty:
                new_c.append(ints.iloc[k])
    
    new_c = gps.GeoSeries(new_c)
    new_cc = gps.GeoDataFrame({'geometry': new_c})
    new_cc.set_geometry("geometry", inplace=True)  # Set the geometry column
    
    bounda = gps.overlay(new_cc, tempPolygonDf, how='difference')
    bounda.set_geometry("geometry", inplace=True)  # Set the geometry column
    
    if output_plot:
        if not bounda.empty:
            fig, ax = plt.subplots(figsize=(10, 10))
            bounda.plot(ax=ax, color='red')
            plt.show()
        else:
            print("Warning: The boundary weight calculation resulted in an empty GeoDataFrame.")
    
    # Change multipolygon to polygon
    if not bounda.empty:
        bounda = bounda.explode()
        bounda.reset_index(drop=True, inplace=True)
    
    return bounda

In [ ]:
def dividePolygonsInTrainingAreas(trainingPolygon, trainingArea):
    '''
    Assign annotated polygons to the training areas.
    '''
    # For efficiency, assigned polygons are removed from the list, we make a copy here. 
    cpTrainingPolygon = trainingPolygon.copy()
    splitPolygons = {}
    for i in tqdm(trainingArea.index):
        spTemp = []
        allocated = []
        for j in cpTrainingPolygon.index:
            if trainingArea.loc[i]['geometry'].intersects(cpTrainingPolygon.loc[j]['geometry']):
                spTemp.append(cpTrainingPolygon.loc[j]['geometry'])
                allocated.append(j)

        # Debug print statement to inspect spTemp
        print(f"Training Area ID: {trainingArea.loc[i]['id']}")
        print(f"spTemp contents: {spTemp}")

        # Order of bounds: minx miny maxx maxy
        boundary = calculateBoundaryWeight(spTemp, scale_polygon=1.5, output_plot=config.show_boundaries_during_processing)
        splitPolygons[trainingArea.loc[i]['id']] = {'polygons': spTemp, 'boundaryWeight': boundary, 'bounds': list(trainingArea.bounds.loc[i]),}
        cpTrainingPolygon = cpTrainingPolygon.drop(allocated)
    return splitPolygons

# areasWithPolygons contains the object polygons and weighted boundaries for each area!
areasWithPolygons = dividePolygonsInTrainingAreas(trainingPolygon, trainingArea)
print(f'Assigned training polygons in {len(areasWithPolygons)} training areas and created weighted boundaries for polygons')


In [ ]:
import os
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

def readInputImages(imageBaseDir, rawImageFileType, rawImagePrefix, target_crs="EPSG:4326"):
    """
    Reads all images with prefix rawImagePrefix and rawImageFileType datatype in the imageBaseDir directory.
    Reprojects the images to the target CRS (default is EPSG:4326).
    """     
    imageFn = []
    for root, dirs, files in os.walk(imageBaseDir):
        for file in files:
            if file.endswith(rawImageFileType) and file.startswith(rawImagePrefix):
                file_path = os.path.join(root, file)
                # Reproject the image to the target CRS
                with rasterio.open(file_path) as src:
                    transform, width, height = calculate_default_transform(
                        src.crs, target_crs, src.width, src.height, *src.bounds
                    )
                    kwargs = src.meta.copy()
                    kwargs.update({
                        'crs': target_crs,
                        'transform': transform,
                        'width': width,
                        'height': height
                    })

                    # Reproject and save the image in memory
                    with rasterio.open(file_path.replace(rawImageFileType, f"_reprojected{rawImageFileType}"), 'w', **kwargs) as dst:
                        for i in range(1, src.count + 1):
                            reproject(
                                source=rasterio.band(src, i),
                                destination=rasterio.band(dst, i),
                                src_transform=src.transform,
                                src_crs=src.crs,
                                dst_transform=transform,
                                dst_crs=target_crs,
                                resampling=Resampling.nearest
                            )
                    imageFn.append(file_path.replace(rawImageFileType, f"_reprojected{rawImageFileType}"))
    return imageFn

# Read and reproject the raw input images
inputImages = readInputImages(config.raw_image_base_dir, config.raw_image_file_type, config.raw_image_prefix)
print(f'Found a total of {len(inputImages)} raw image(s) to process!')

# Print the file paths of the read images
for img in inputImages:
    print(f'Read image: {img}')

In [ ]:
# For each raw satellite image, determine if it overlaps with a training area. 
# If a overlap if found, then extract + write the overlapping part of the raw image, create + write an image from training polygons and create + write an image from boundary weights in the that overlapping region.

def drawPolygons(polygons, shape, outline, fill):
    """
    From the polygons, create a numpy mask with fill value in the foreground and 0 value in the background.
    Outline (i.e the edge of the polygon) can be assigned a separate value.
    """
    mask = np.zeros(shape, dtype=np.uint8)
    mask = PIL.Image.fromarray(mask)
    draw = PIL.ImageDraw.Draw(mask)
    #Syntax: PIL.ImageDraw.Draw.polygon(xy, fill=None, outline=None)
    #Parameters:
    #xy – Sequence of either 2-tuples like [(x, y), (x, y), …] or numeric values like [x, y, x, y, …].
    #outline – Color to use for the outline.
    #fill – Color to use for the fill.
    #Returns: An Image object.
    for polygon in polygons:
        xy = [(point[1], point[0]) for point in polygon]
        draw.polygon(xy=xy, outline=outline, fill=fill)
    mask = np.array(mask)#, dtype=bool)   
    return(mask)
    

# my original
#def rowColPolygons(areaDf, areaShape, profile, filename, outline, fill):
    """
    Convert polygons coordinates to image pixel coordinates, create annotation image using drawPolygons() 
    and write the results into an image file.
    """
#    transform = profile['transform']
#    polygons = []
    
#    for i in areaDf.index:
#        geom = areaDf.loc[i]['geometry']
        
#        if geom.geom_type == 'Polygon':
            # Handle Polygon type
#            a, b = zip(*list(geom.exterior.coords))
#            row, col = rasterio.transform.rowcol(transform, a, b)
#            polygons.append(list(zip(row, col)))
        
#        elif geom.geom_type == 'MultiPolygon':
            # Handle MultiPolygon type
#            for polygon in geom.geoms:
#                a, b = zip(*list(polygon.exterior.coords))
#                row, col = rasterio.transform.rowcol(transform, a, b)
#                polygons.append(list(zip(row, col)))
    
    # Write polygons to JSON file
#    with open(filename, 'w') as outfile:  
#        json.dump({'Trees': polygons}, outfile)
        
    # Create and save mask image
#    mask = drawPolygons(polygons, areaShape, outline=outline, fill=fill)
#    profile['dtype'] = rasterio.int16
#    with rasterio.open(filename.replace('json', 'png'), 'w', **profile) as dst:
#        dst.write(mask.astype(rasterio.int16), 1)

#AI cursor:
def rowColPolygons(areaDf, areaShape, profile, filename, outline, fill):
    """
    Convert polygons coordinates to image pixel coordinates, create annotation image using drawPolygons() 
    and write the results into an image file.
    """
    transform = profile['transform']
    polygons = []
    
    for i in areaDf.index:
        geom = areaDf.loc[i]['geometry']
        
        if geom.geom_type == 'Polygon':
            # Handle Polygon type
            a, b = zip(*list(geom.exterior.coords))
            row, col = rasterio.transform.rowcol(transform, a, b)
            # Convert numpy.int32 to regular Python int
            row = [int(r) for r in row]
            col = [int(c) for c in col]
            polygons.append(list(zip(row, col)))
        
        elif geom.geom_type == 'MultiPolygon':
            # Handle MultiPolygon type
            for polygon in geom.geoms:
                a, b = zip(*list(polygon.exterior.coords))
                row, col = rasterio.transform.rowcol(transform, a, b)
                # Convert numpy.int32 to regular Python int
                row = [int(r) for r in row]
                col = [int(c) for c in col]
                polygons.append(list(zip(row, col)))
    
    # Write polygons to JSON file
    with open(filename, 'w') as outfile:  
        json.dump({'Trees': polygons}, outfile)
        
    # Create and save mask image
    mask = drawPolygons(polygons, areaShape, outline=outline, fill=fill)
    profile['dtype'] = rasterio.int16
    with rasterio.open(filename.replace('json', 'png'), 'w', **profile) as dst:
        dst.write(mask.astype(rasterio.int16), 1)
        
###not working
#def rowColPolygons(areaDf, areaShape, profile, filename, outline, fill):
    """
    Convert polygons coordinates to image pixel coordinates, create annotation image using drawPolygons(), 
    and write the results into an image file.
    """
#    transform = profile.get('transform')
#    if transform is None:
#        raise ValueError("Transform is missing in the profile.")

    # Validate geometries
#    areaDf = areaDf[areaDf.is_valid]
#    areaDf['geometry'] = areaDf['geometry'].apply(make_valid)

#    polygons = []
#    for i in areaDf.index:
#        geom = areaDf.loc[i]['geometry']

#        try:
#            if geom.geom_type == 'Polygon':
#                a, b = zip(*list(geom.exterior.coords))
#                row, col = rasterio.transform.rowcol(transform, a, b)
#                polygons.append(list(zip(row, col)))

#            elif geom.geom_type == 'MultiPolygon':
#                for polygon in geom.geoms:
#                    a, b = zip(*list(polygon.exterior.coords))
#                    row, col = rasterio.transform.rowcol(transform, a, b)
#                    polygons.append(list(zip(row, col)))

#        except Exception as e:
#            print(f"Invalid geometry at index {i}. Skipping... Error: {e}")
#            continue

#    # Write polygons to JSON file
#    with open(filename, 'w') as outfile:  
#        json.dump({'Trees': polygons}, outfile)

    # Create and save mask image
#    mask = drawPolygons(polygons, areaShape, outline=outline, fill=fill)
#    profile['dtype'] = rasterio.int16
#    with rasterio.open(filename.replace('json', 'png'), 'w', **profile) as dst:
#        dst.write(mask.astype(rasterio.int16), 1)

#AI cursor
def writeExtractedImageAndAnnotation(img, sm, profile, polygonsInAreaDf, boundariesInAreaDf, writePath, imagesFilename, annotationFilename, boundaryFilename, bands, writeCounter, normalize=True):
    """
    Write the part of raw image that overlaps with a training area into a separate image file. 
    Use rowColPolygons to create and write annotation and boundary image from polygons in the training area.
    """
    try:
        # Print debug information
        print(f"Writing to directory: {writePath}")
        print(f"Processing counter: {writeCounter}")
        
        # Create directory if it doesn't exist
        os.makedirs(writePath, exist_ok=True)
        
        # Ensure 'geometry' column is explicitly set for polygonsInAreaDf
        if 'geometry' not in polygonsInAreaDf.columns:
            polygonsInAreaDf = gps.GeoDataFrame(polygonsInAreaDf, geometry=polygonsInAreaDf[0])
        
        # Set the CRS if it's missing
        if polygonsInAreaDf.crs is None:
            polygonsInAreaDf.set_crs(epsg=4326, inplace=True)
        
        if boundariesInAreaDf.crs is None:
            boundariesInAreaDf.set_crs(epsg=4326, inplace=True)
        
        # Ensure the CRS of polygonsInAreaDf matches the image profile CRS
        if polygonsInAreaDf.crs != profile['crs']:
            polygonsInAreaDf = polygonsInAreaDf.to_crs(profile['crs'])
        
        if boundariesInAreaDf.crs != profile['crs']:
            boundariesInAreaDf = boundariesInAreaDf.to_crs(profile['crs'])

        # Write image files
        for band, imFn in zip(bands, imagesFilename):
            output_path = os.path.join(writePath, f"{imFn}_{writeCounter}.png")
            print(f"Writing image to: {output_path}")
            
            dt = sm[0][band].astype(profile['dtype'])
            if normalize:
                dt = image_normalize(dt, axis=None)
            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(dt, 1)

        # Write annotation file
        if annotationFilename and not polygonsInAreaDf.empty:
            annotation_json_path = os.path.join(writePath, f"{annotationFilename}_{writeCounter}.json")
            print(f"Writing annotation to: {annotation_json_path}")
            
            rowColPolygons(
                polygonsInAreaDf, 
                (sm[0].shape[1], sm[0].shape[2]), 
                profile, 
                annotation_json_path, 
                outline=0, 
                fill=1
            )
        
        # Write boundary file
        if boundaryFilename and not boundariesInAreaDf.empty:
            boundary_path = os.path.join(writePath, f"{boundaryFilename}_{writeCounter}.json")
            print(f"Writing boundary to: {boundary_path}")
            
            rowColPolygons(
                boundariesInAreaDf, 
                (sm[0].shape[1], sm[0].shape[2]),
                profile, 
                boundary_path, 
                outline=1, 
                fill=1
            )
        
        return writeCounter + 1
    
    except Exception as e:
        print(f"Error in writeExtractedImageAndAnnotation: {str(e)}")
        print(f"Error type: {type(e).__name__}")
        import traceback
        print(f"Full traceback: {traceback.format_exc()}")
        return writeCounter

        
#def writeExtractedImageAndAnnotation(img, sm, profile, polygonsInAreaDf, boundariesInAreaDf, writePath, imagesFilename, annotationFilename, boundaryFilename, bands, writeCounter, normalize=True):
    """
    Write the part of raw image that overlaps with a training area into a separate image file. 
    Use rowColPolygons to create and write annotation and boundary image from polygons in the training area.
    """
#    try:
#        # Ensure 'geometry' column is explicitly set for polygonsInAreaDf
#        if 'geometry' not in polygonsInAreaDf.columns:
#            polygonsInAreaDf = gps.GeoDataFrame(polygonsInAreaDf, geometry=polygonsInAreaDf[0])
        
        # Set the CRS if it's missing
#        if polygonsInAreaDf.crs is None:
#            polygonsInAreaDf.set_crs(epsg=4326, inplace=True)  # Adjust EPSG code if necessary
        
#        if boundariesInAreaDf.crs is None:
#            boundariesInAreaDf.set_crs(epsg=4326, inplace=True)  # Adjust EPSG code if necessary
        
        # Ensure the CRS of polygonsInAreaDf matches the image profile CRS
#        if polygonsInAreaDf.crs != profile['crs']:
#            polygonsInAreaDf = polygonsInAreaDf.to_crs(profile['crs'])
        
        # Ensure the CRS of boundariesInAreaDf matches the image profile CRS
#        if boundariesInAreaDf.crs != profile['crs']:
#            boundariesInAreaDf = boundariesInAreaDf.to_crs(profile['crs'])
            
        # Ensure profile keys are JSON-serializable
        #profile = {key: ensure_serializable(value) for key, value in profile.items()} ###new

#        for band, imFn in zip(bands, imagesFilename):
            # Rasterio reads file channel first, so the sm[0] has the shape [1 or ch_count, x,y]
            # If raster has multiple channels, then bands will be [0, 1, ...] otherwise simply [0]
#            dt = sm[0][band].astype(profile['dtype'])
#            if normalize:
#                dt = image_normalize(dt, axis=None)
#            with rasterio.open(os.path.join(writePath, imFn+'_{}.png'.format(writeCounter)), 'w', **profile) as dst:
#                dst.write(dt, 1) 

#        if annotationFilename:
#            annotation_json_filepath = os.path.join(writePath, annotationFilename+'_{}.json'.format(writeCounter))
#            rowColPolygons(polygonsInAreaDf, (sm[0].shape[1], sm[0].shape[2]), profile, annotation_json_filepath, outline=0, fill=1)
        
#        if boundaryFilename:
#            boundary_json_filepath = os.path.join(writePath, boundaryFilename+'_{}.json'.format(writeCounter))
#            rowColPolygons(boundariesInAreaDf, (sm[0].shape[1], sm[0].shape[2]), profile, boundary_json_filepath, outline=1, fill=1)
        
#        return writeCounter + 1
    
#    except Exception as e:
#        print(e)
#        print("Something nasty happened, could not write the annotation or the mask file!")
 #       return writeCounter


In [ ]:
def findOverlap(img, areasWithPolygons, writePath, imageFilename, annotationFilename, boundaryFilename, bands, writeCounter=1):
    """
    Finds overlap of image with a training area.
    Use writeExtractedImageAndAnnotation() to write the overlapping training area and corresponding polygons in separate image files.
    """
    overlapppedAreas = set()
    for areaID, areaInfo in areasWithPolygons.items():
        polygonsInAreaDf = gps.GeoDataFrame(areaInfo['polygons'])
        boundariesInAreaDf = gps.GeoDataFrame(areaInfo['boundaryWeight'])    
        bboxArea = box(*areaInfo['bounds'])
        bboxImg = box(*img.bounds)
        if bboxArea.intersects(bboxImg):
            profile = img.profile  
            sm = rasterio.mask.mask(img, [bboxArea], all_touched=True, crop=True)
            profile['height'] = sm[0].shape[1]
            profile['width'] = sm[0].shape[2]
            profile['transform'] = sm[1]
            profile['blockxsize'] = 32
            profile['blockysize'] = 32
            profile['count'] = 1
            profile['dtype'] = rasterio.float32
            writeCounter = writeExtractedImageAndAnnotation(img, sm, profile, polygonsInAreaDf, boundariesInAreaDf, writePath, imageFilename, annotationFilename, boundaryFilename, bands, writeCounter)
            overlapppedAreas.add(areaID)
    return writeCounter, overlapppedAreas


In [ ]:
def extractAreasThatOverlapWithTrainingData(inputImages, areasWithPolygons, writePath, ndviFilename, panFilename, annotationFilename, boundaryFilename, bands, writeCounter):
    """
    Iterates over raw images and uses findOverlap() to extract areas that overlap with training data. 
    """
    if not os.path.exists(writePath):
        os.makedirs(writePath)
        
    overlapppedAreas = set()                   
    for imgFile in tqdm(inputImages):
        img = rasterio.open(imgFile)
        writeCounter, imgOverlapppedAreas = findOverlap(img, areasWithPolygons, writePath=writePath, imageFilename=[ndviFilename, panFilename], annotationFilename=annotationFilename, boundaryFilename=boundaryFilename, bands=bands, writeCounter=writeCounter)
        overlapppedAreas.update(imgOverlapppedAreas)

    print(f"Total areas with overlaps: {len(overlapppedAreas)}")
    return writeCounter

In [ ]:
bands = [0, 1]
writeCounter = 0
writeCounter = extractAreasThatOverlapWithTrainingData(
    inputImages, areasWithPolygons, config.path_to_write,  # Use path_to_write
    config.extracted_ndvi_filename, config.extracted_pan_filename, 
    config.extracted_annotation_filename, config.extracted_boundary_filename, 
    bands, writeCounter
)


In [ ]:
#from PIL import Image

# Disable the decompression bomb protection
Image.MAX_IMAGE_PIXELS = None

# Display extracted image
sampleImage = '_0.png'
fn = os.path.join(config.path_to_write, config.extracted_ndvi_filename + sampleImage )

ndvi_img = Image.open(fn)
pan_img = Image.open(fn.replace(config.extracted_ndvi_filename ,config.extracted_pan_filename))
read_ndvi_img = np.array(ndvi_img)
read_pan_img = np.array(pan_img)
annotation_im = Image.open(fn.replace(config.extracted_ndvi_filename ,config.extracted_annotation_filename))
read_annotation = np.array(annotation_im)
weight_im = Image.open(fn.replace(config.extracted_ndvi_filename ,config.extracted_boundary_filename))
read_weight = np.array(weight_im)
all_images = np.array([read_ndvi_img, read_pan_img, read_annotation, read_weight ])

#display_images(np.expand_dims(np.transpose(all_images, axes=(1,2,0)), axis=0))
# plt.imshow(read_weight)

# Check if any images (NDVI, PAN, annotation, weight) are empty (all zeros or None)
def check_and_display_images(ndvi_img, pan_img, annotation_img, weight_img):
    # Convert images to arrays
    ndvi_arr = np.array(ndvi_img)
    pan_arr = np.array(pan_img)
    annotation_arr = np.array(annotation_img)
    weight_arr = np.array(weight_img)

    # Check for empty or None images
    if np.all(ndvi_arr == 0):
        print("NDVI image is empty (all zeros)!")
    elif ndvi_arr is None:
        print("NDVI image is None!")
    else:
        print(f"NDVI image shape: {ndvi_arr.shape}")
    
    if np.all(pan_arr == 0):
        print("PAN image is empty (all zeros)!")
    elif pan_arr is None:
        print("PAN image is None!")
    else:
        print(f"PAN image shape: {pan_arr.shape}")

    if np.all(annotation_arr == 0):
        print("Annotation image is empty (all zeros)!")
    elif annotation_arr is None:
        print("Annotation image is None!")
    else:
        print(f"Annotation image shape: {annotation_arr.shape}")

    if np.all(weight_arr == 0):
        print("Weight image is empty (all zeros)!")
    elif weight_arr is None:
        print("Weight image is None!")
    else:
        print(f"Weight image shape: {weight_arr.shape}")

    # Overlay of annotation with boundary
    overlay = annotation_arr + weight_arr
    overlay = overlay[:, :, np.newaxis]

    # Display images (ndvi, pan, annotation, weight, overlay)
    all_images = np.stack((ndvi_arr, pan_arr, annotation_arr, weight_arr), axis=-1)
    #display_images(np.concatenate((all_images, overlay), axis=-1))

# Call the function
check_and_display_images(ndvi_img, pan_img, annotation_im, weight_im)


In [ ]:
# Disable the decompression bomb protection
Image.MAX_IMAGE_PIXELS = None

def check_image_properties(image_path):
    """
    Check image properties without loading the full image into memory
    """
    try:
        with Image.open(image_path) as img:
            print(f"\nFile: {os.path.basename(image_path)}")
            print(f"Size: {img.size}")
            print(f"Mode: {img.mode}")
            
            # Check if image is empty (sample a few pixels)
            sample = np.array(img.crop((0, 0, min(100, img.size[0]), min(100, img.size[1]))))
            if np.all(sample == 0):
                print("Warning: First 100x100 pixels are all zeros!")
            
            return True
    except Exception as e:
        print(f"Error checking {os.path.basename(image_path)}: {str(e)}")
        return False

def safe_check_images():
    try:
        # Define file paths
        sampleImage = '_0.png'
        base_path = config.path_to_write
        
        paths = {
            'NDVI': os.path.join(base_path, f"{config.extracted_ndvi_filename}{sampleImage}"),
            'PAN': os.path.join(base_path, f"{config.extracted_pan_filename}{sampleImage}"),
            'Annotation': os.path.join(base_path, f"{config.extracted_annotation_filename}{sampleImage}"),
            'Weight': os.path.join(base_path, f"{config.extracted_boundary_filename}{sampleImage}")
        }
        
        print("Checking images in:")
        for name, path in paths.items():
            print(f"{name}: {path}")
            if not os.path.exists(path):
                print(f"File not found: {path}")
                return
        
        # Check each image's properties - Fixed this line
        all_valid = all(check_image_properties(path) for path in paths.values())
        
        if all_valid:
            print("\nAll images are valid and accessible!")
            
            # Load and display a small sample region
            sample_size = 1000  # adjust this value based on your memory constraints
            print(f"\nLoading {sample_size}x{sample_size} sample region for visualization...")
            
            samples = {}
            for name, path in paths.items():
                with Image.open(path) as img:
                    # Take a sample from the center of the image
                    center_x = img.size[0] // 2
                    center_y = img.size[1] // 2
                    box = (
                        center_x - sample_size//2,
                        center_y - sample_size//2,
                        center_x + sample_size//2,
                        center_y + sample_size//2
                    )
                    samples[name] = np.array(img.crop(box))
            
            print("Sample region shapes:")
            for name, sample in samples.items():
                print(f"{name}: {sample.shape}")
            
            # Create overlay and display all images
            try:
                overlay = samples['Annotation'] + samples['Weight']
                
                # Create figure with subplots
                plt.figure(figsize=(20, 4))
                
                # Plot NDVI
                plt.subplot(151)
                plt.imshow(samples['NDVI'], cmap='viridis')
                plt.title('NDVI')
                plt.colorbar()
                
                # Plot PAN
                plt.subplot(152)
                plt.imshow(samples['PAN'], cmap='gray')
                plt.title('PAN')
                plt.colorbar()
                
                # Plot Annotation
                plt.subplot(153)
                plt.imshow(samples['Annotation'], cmap='binary')
                plt.title('Annotation')
                plt.colorbar()
                
                # Plot Weight
                plt.subplot(154)
                plt.imshow(samples['Weight'], cmap='binary')
                plt.title('Weight')
                plt.colorbar()
                
                # Plot Overlay
                plt.subplot(155)
                plt.imshow(overlay, cmap='binary')
                plt.title('Overlay')
                plt.colorbar()
                
                plt.tight_layout()
                plt.show()
                
                print("\nSuccessfully displayed all images!")
                
            except Exception as e:
                print(f"Error creating display: {str(e)}")
        
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        import traceback
        print(f"Full traceback: {traceback.format_exc()}")

# Call the function
safe_check_images()